<a href="https://colab.research.google.com/github/wyattae/cosc-650-applied-llm-systems/blob/WE_1/src/week1/week1_tokenization_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 (starter): Tokenization Analysis

This is the starter notebook for the Week 1 assignment. It runs as-is on placeholder text so you can see the shape of each step; your job is to replace the placeholders with your own passages and analysis, then commit it to your repository and open a pull request.

Cells marked **TODO (you)** are where you do the work. Everything runs in Jupyter or Google Colab. No GPU, no API key, one dependency: `tiktoken`.

The five parts match the assignment: stand up your repo, run the analysis, evaluate with real figures, find one failure, and submit.

In [ ]:
# Setup. In Colab, uncomment the install line on first run.
# !pip install tiktoken
import os, pathlib
os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.') / '.tiktoken_cache').resolve())
os.makedirs(os.environ['TIKTOKEN_CACHE_DIR'], exist_ok=True)

import tiktoken
gpt4  = tiktoken.get_encoding('cl100k_base')   # GPT-4 / GPT-3.5
gpt4o = tiktoken.get_encoding('o200k_base')    # GPT-4o
print('tiktoken', tiktoken.__version__, '- encoders ready (cl100k_base, o200k_base)')

## Part 1: Stand up your repository

Do this once, outside the notebook:

1. Create a public repo (suggested name `cosc-650`).
2. Add a `README.md` a stranger could read (what it is, how it is organized, the tools you use).
3. Add an agent context file that your AI tool reads, with project context and conventions. `AGENTS.md` is the cross-tool convention; `CLAUDE.md` and `GEMINI.md` are tool-specific variants. Use whichever your tool reads.
4. Work on a branch and open a pull request into `main`. You will do this every week.

Then commit this notebook into the repo and keep going.

## Helpers (provided)

Two small functions: count tokens for a string, and show the exact sub-token pieces a word breaks into. The demo uses a line you may recognize.

In [ ]:
def count_tokens(text, enc):
    return len(enc.encode(text))

def show_split(word, enc=gpt4):
    ids = enc.encode(word)
    pieces = [enc.decode([i]) for i in ids]
    print(f'{word!r:18s} -> {len(ids)} token(s): {pieces}')

# demo: some short strings are a single token; capitalized or rarer words fragment
for w in ['Panic', ' towel', '42', 'antidisestablishmentarianism']:
    show_split(w)

## Part 2: Your passages

**TODO (you):** replace the two placeholders with your own text. The non-English passage must be at least 100 words, with a faithful English translation. The placeholders below are short Hitchhiker's Guide lines so the notebook runs; swap in your real passages.

In [ ]:
# TODO (you): replace both with your own >=100-word passage and its translation.
english_text = "Don't panic, and always know where your towel is."
foreign_text = "Pas de panique, et sachez toujours o\u00f9 se trouve votre serviette."

print('English words:', len(english_text.split()))
print('Foreign words:', len(foreign_text.split()))

def report(label, text):
    print(f'{label:9s} | chars {len(text):4d} | GPT-4 {count_tokens(text, gpt4):4d} | GPT-4o {count_tokens(text, gpt4o):4d}')

report('English', english_text)
report('Foreign', foreign_text)

tax_gpt4  = count_tokens(foreign_text, gpt4)  / count_tokens(english_text, gpt4)
tax_gpt4o = count_tokens(foreign_text, gpt4o) / count_tokens(english_text, gpt4o)
print(f'\nMultilingual tax  GPT-4: {tax_gpt4:.2f}x   GPT-4o: {tax_gpt4o:.2f}x')
# TODO (you): one or two sentences interpreting these numbers for YOUR language pair.

## Part 3: Evaluate with real figures

Turn the counts into engineering consequences. The skeleton below computes both; keep it pointed at your real passages.

In [ ]:
CTX = 128_000
en = count_tokens(english_text, gpt4)
fo = count_tokens(foreign_text, gpt4)
print(f'A {CTX:,}-token window holds about {CTX//en:,} English copies and {CTX//fo:,} foreign copies of your passage.')
print(f'Per-request cost multiplier for the foreign language: {fo/en:.2f}x (billing is per token).')
# TODO (you): state what this means for a product serving users in your chosen language.

## Part 4: Bias splits and one failure

**TODO (you):** (a) pick three words where your non-English form fragments far worse than the English equivalent, and show both with `show_split`; (b) find ONE input whose token count defies intuition and explain it. A few failure candidates are demonstrated below to get you started; replace them with your own find and write the explanation plus a mitigation.

In [ ]:
# (a) TODO (you): three real bias pairs from your languages.
print('English baselines:')
for w in ['water', 'friendship', 'dignity']:
    show_split(w)

print('\n(b) failure candidates to explore (replace with your own find):')
show_split('\U0001F680')                 # a rocket emoji
show_split('hello')                       # baseline
show_split(' hello')                      # a leading space changes the tokenization
show_split('https://www.example.com')     # URLs fragment
# TODO (you): explain WHY your chosen case behaves this way, and how you would budget or normalize around it.

## Part 5: Submit

Before you open the pull request, check:

- The notebook runs top to bottom on **your** passages, not the placeholders.
- Your three bias splits are shown and explained.
- The failure case has a cause and a mitigation.
- The PR description has a one-paragraph result summary with your headline numbers.
- You linked one issue in your repo logging this as a research note (title, inputs, what you found).

Rubric: repo quality (15), counts from both tokenizers (20), tax computed (15), three bias splits (20), cost and context figures (15), the failure case (10), PR hygiene (5).